# MorphoFeatures Tutorial

This notebook demonstrates the refactored MorphoFeatures pipeline. The most important thing to keep straight is that an **embedding matrix is not the raw input data**. It is the feature table produced by a trained texture or shape encoder. Downstream analysis starts from that feature table.

## 0. What Is My Input Data?

There are three common starting points:

1. **You already have embeddings**: use an existing file such as `analysis/data/morphofeatures_all_cells.npy` and go directly to UMAP, clustering, classification, or plotting.
2. **You have raw texture data**: provide raw EM volume data, cell segmentation, nucleus segmentation, cell-to-nucleus mapping, and metadata tables. The texture encoder turns these into an embedding matrix.
3. **You have shape data**: provide point-cloud arrays for each cell, optional per-point features such as normals, and optional cell IDs. The shape encoder turns these into an embedding matrix.

The data flow is:

```text
texture raw data -> texture model -> embedding matrix -> analysis
shape point clouds -> shape model -> embedding matrix -> analysis
precomputed embedding matrix -------------------------> analysis
```

An embedding matrix always has one row per cell. The first column is the cell `label_id`; the rest are learned morphology features.

## 1. Configuration

Pipeline behavior is controlled by YAML files in `configs/`. Paths, volume constants, model settings, loader settings, and analysis parameters should live in config rather than inside Python code.

In [ ]:
from pathlib import Path
from morphofeatures.config.loading import load_config

config = load_config(Path('../configs/analysis.yaml'), Path('../configs/default.yaml'))
config.keys()

## 2. Embedding Data Model

All downstream analysis expects an embedding matrix where the first column is `label_id` and the remaining columns are numeric features. This matrix comes from `morphofeatures-texture predict`, `morphofeatures-shape embed`, or from precomputed files in this repository. The helper functions sort by ID and validate alignment when multiple feature files are merged.

In [ ]:
import numpy as np
from morphofeatures.data.embeddings import EmbeddingTable, merge_embedding_tables

toy = EmbeddingTable(ids=np.array([2, 1]), features=np.array([[0.2, 0.3], [0.1, 0.4]]))
toy.as_matrix()

To load real repository data, point the reader at one or more precomputed embedding files from `analysis/data/` or `data_mobie/`. For example, `analysis/data/morphofeatures_all_cells.npy` is already an embedding matrix from the paper workflow.

In [ ]:
embedding_path = Path('../analysis/data/morphofeatures_all_cells.npy')
if embedding_path.exists():
    merged = merge_embedding_tables([embedding_path], scale=True)
    print(merged.ids.shape, merged.features.shape)
else:
    print('Example data file not found in this checkout.')

## 3. Shape Feature Extraction

Shape models consume point-cloud arrays from a `.npz` with keys `points`, optional `features`, and optional `ids`. This is raw input to the shape pipeline. During training, the loader returns two augmented views per cell for contrastive metric learning. During inference, it returns one view per cell and writes the embedding matrix: `label_id + learned feature vector`.

In [ ]:
# GPU/dependency-dependent example:
# !morphofeatures-train shape --config ../configs/shape_train.yaml --dry-run
# !morphofeatures-train shape --config ../configs/shape_train.yaml
# After training, edit ../configs/shape_inference.yaml so model.checkpoint points to the trained checkpoint.
# !morphofeatures-shape embed --config ../configs/shape_inference.yaml --save-to ../runs/shape_embeddings.npy

For a new model or dataset, create a new config with the point-cloud paths, update `model.kwargs` if the feature dimensionality changes, and keep the output embedding format unchanged.

## 4. Texture Feature Extraction

Texture models read raw EM volumes plus cell/nucleus segmentations and metadata tables. This is raw input to the texture pipeline. The important dataset-specific values are now config fields: raw container, BDV segmentation XMLs, table paths, voxel resolution, reference high-resolution shape, crop size, and transform settings. Prediction writes the embedding matrix used by the analysis steps.

In [ ]:
# GPU/dependency-dependent examples:
# !morphofeatures-train texture /path/to/texture_experiment --dry-run
# !morphofeatures-train texture /path/to/texture_experiment --devices 0
# !morphofeatures-texture predict /path/to/texture_experiment --devices 0
# !morphofeatures-texture predict /path/to/texture_experiment --save-patches --aggregate-patches

For unseen raw data, copy `configs/texture_train.yaml`, replace the `data.paths` or legacy `root_dir/version` fields, verify the metadata columns (`label_id`, `bb_min_*`, `bb_max_*`, `anchor_*`), then train or run inference.

## 5. Downstream Analysis

Analysis modules are pure functions around explicit inputs. This makes it easier to reuse them in scripts, notebooks, and tests.

In [ ]:
from morphofeatures.analysis.clustering import save_labels

ids = np.array([1, 2, 3])
labels = np.array([0, 1, 1])
umap_embedding = np.array([[0.0, 0.1], [1.0, 1.1], [1.2, 0.9]])
Path('../runs').mkdir(exist_ok=True)
cluster_table = save_labels(ids, labels, umap_embedding, Path('../runs/tutorial_clusters.tsv'))
cluster_table.head()

Common CLI commands:

```bash
morphofeatures-analysis classify analysis/data/morphofeatures_all_cells.npy
morphofeatures-analysis cluster analysis/data/morphofeatures_all_cells.npy --save-path runs/clusters.tsv
morphofeatures-analysis bilateral analysis/data/morphofeatures_all_cells.npy --save-dist runs/bilateral.tsv
morphofeatures-analysis genes runs/clusters.tsv --one-gene foxA
```